# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadTalha-pk/ML-Internship-Task1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One warehouse row represents one pseudonymized content item for one client on one report date. For my modeling frame, I will later aggregate these daily rows so that one modeling row represents one content page at a prediction date.
Time window: For this assignment, March 2026 will be the feature window. The future outcome will be measured in April 2026. All model features must be known by the end of March, so April information will never be used as a feature.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: I will build exactly five features from March 2026 search-performance data: average daily impressions, average daily clicks, CTR, average position, and number of days with impressions.
Label / proxy: My provisional future-decline proxy will be calculated from the following month. A page will be labelled as declining when its average daily impressions in April are at least 30% lower than in March. This threshold is provisional and can be tested later.
Context: client_hash_id and content_hash_id are used only for grouping, joining, and validation. They will not be model features.
Excluded: April performance values are excluded from the feature set because April is the future outcome window. Using them as features would leak the answer into the model.
Output: A ranked list of pages by estimated future-decline risk for human review.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [28]:
!pip -q install -U duckdb huggingface_hub

import os
import duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    PROVIDER credential_chain
)
""")

MARCH = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
"""

print("Warehouse connection ready.")

Warehouse connection ready.


In [29]:
grain_check = con.sql(f"""
SELECT COUNT(*) AS duplicate_grain_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {MARCH}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_grain_groups
0,0


In [30]:
result = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {MARCH}
""").df()

result

,rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [31]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows

FROM {MARCH}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


Verification summary: The March 2026 warehouse slice contains 9,841,378 daily rows covering 1 March to 31 March 2026. The grain check returned 0 duplicate date-client-content groups, so the stated daily grain holds. Search Console data is available on 3,611,061 rows, GA4 data on 413,966 rows, and both are available on 364,347 rows. I will use the availability flags rather than treating unavailable measurements as genuine zeros.

In [32]:
feature_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    AVG(gsc_impressions) AS avg_daily_impressions_mar,
    AVG(gsc_clicks) AS avg_daily_clicks_mar,

    100.0 * SUM(gsc_clicks)
        / NULLIF(SUM(gsc_impressions), 0)
        AS ctr_mar_pct,

    AVG(gsc_avg_position) AS avg_position_mar,

    COUNT(*) FILTER (
        WHERE gsc_impressions > 0
    ) AS days_with_impressions_mar

FROM {MARCH}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING
    SUM(gsc_impressions) >= 100
    AND COUNT(*) >= 20
""").df()

print("Feature-frame rows:", len(feature_df))
feature_df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 93812


,client_hash_id,content_hash_id,avg_daily_impressions_mar,avg_daily_clicks_mar,ctr_mar_pct,avg_position_mar,days_with_impressions_mar
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,210.419355,0.225806,0.107313,7.209549,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,14.612903,0.000000,0.000000,2.987198,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,181.612903,0.193548,0.106572,6.724039,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,159.483871,0.419355,0.262945,7.244844,31
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,13.838710,0.032258,0.233100,4.209227,31
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,7.193548,0.032258,0.448430,9.445635,31
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,10.129032,0.032258,0.318471,9.155335,31
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,248.677419,0.645161,0.259437,5.258331,31
8,client_73cda7b4e4f265ea,content_20403327d8d9374c,114.870968,0.322581,0.280820,8.834415,31
9,client_73cda7b4e4f265ea,content_f71459b346aba398,6.034483,0.034483,0.571429,35.609168,29


Five features
avg_daily_impressions_mar — available at the decision moment because it uses only March impressions.
avg_daily_clicks_mar — available because it uses only March clicks.
ctr_mar_pct — available because it is calculated only from March clicks and impressions.
avg_position_mar — available because it uses only March search-position data.
days_with_impressions_mar — available because it counts only March days with observed impressions.
client_hash_id and content_hash_id are context fields only and will not be used as model features.

In [33]:
APRIL = """
read_parquet(
  'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet'
)
"""

april_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    AVG(gsc_impressions) AS avg_daily_impressions_apr,

    COUNT(*) AS measured_days_apr

FROM {APRIL}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

model_df = feature_df.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Require enough April observations
model_df = model_df[
    model_df["measured_days_apr"] >= 20
].copy()

# Future decline = April impressions at least 30% lower than March
model_df["future_decline_proxy"] = (
    model_df["avg_daily_impressions_apr"]
    < 0.70 * model_df["avg_daily_impressions_mar"]
).astype(int)

print("Eligible modeling rows:", len(model_df))
print(
    "Future-decline rate:",
    round(model_df["future_decline_proxy"].mean(), 3)
)

model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "avg_daily_impressions_mar",
        "avg_daily_impressions_apr",
        "future_decline_proxy"
    ]
].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible modeling rows: 88941
Future-decline rate: 0.399


,client_hash_id,content_hash_id,avg_daily_impressions_mar,avg_daily_impressions_apr,future_decline_proxy
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,210.419355,226.233333,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,14.612903,13.500000,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,181.612903,282.500000,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,159.483871,203.033333,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,13.838710,9.566667,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,7.193548,18.400000,0
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,10.129032,10.733333,0
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,248.677419,175.366667,0
8,client_73cda7b4e4f265ea,content_20403327d8d9374c,114.870968,139.900000,0
9,client_73cda7b4e4f265ea,content_f71459b346aba398,6.034483,4.592593,0


Future outcome proxy: The March features are known before the future outcome period begins. I use April as the later outcome window. For this assignment, future_decline_proxy = 1 when a page's average daily impressions in April are at least 30% lower than in March; otherwise it is 0. The 30% threshold is provisional and will be tested later.

In [34]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import numpy as np

# The five honest March features
honest_features = [
    "avg_daily_impressions_mar",
    "avg_daily_clicks_mar",
    "ctr_mar_pct",
    "avg_position_mar",
    "days_with_impressions_mar"
]

# Deliberately create a LEAK using April outcome information
model_df["apr_to_mar_ratio_LEAK"] = (
    model_df["avg_daily_impressions_apr"]
    / model_df["avg_daily_impressions_mar"]
)

X_groups = model_df["client_hash_id"]
y = model_df["future_decline_proxy"]

# Keep whole clients together
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(model_df, y, groups=X_groups)
)

def quick_auc(columns):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(
            max_depth=3,
            class_weight="balanced",
            random_state=42
        )
    )

    model.fit(
        model_df.iloc[train_idx][columns],
        y.iloc[train_idx]
    )

    probabilities = model.predict_proba(
        model_df.iloc[test_idx][columns]
    )[:, 1]

    return roc_auc_score(
        y.iloc[test_idx],
        probabilities
    )

honest_auc = quick_auc(honest_features)

leaky_auc = quick_auc(
    honest_features + ["apr_to_mar_ratio_LEAK"]
)

print("Honest ROC-AUC:", round(honest_auc, 3))
print("Leaky ROC-AUC:", round(leaky_auc, 3))

Honest ROC-AUC: 0.608
Leaky ROC-AUC: 1.0


In [35]:
model_df.drop(
    columns=["apr_to_mar_ratio_LEAK"],
    inplace=True
)

print("Leak removed.")
print("Honest ROC-AUC kept:", round(honest_auc, 3))
print(
    "Leak still present:",
    "apr_to_mar_ratio_LEAK" in model_df.columns
)

Leak removed.
Honest ROC-AUC kept: 0.608
Leak still present: False


Leakage lesson: The honest model achieved a ROC-AUC of 0.608 using only information available in March. After I deliberately added apr_to_mar_ratio_LEAK, which contains April outcome information, the ROC-AUC increased to 1.000. This perfect-looking score is not valid because the model was given future information that is directly related to the target. I removed the leaked feature and kept the honest result. This does not mean the honest model is 60.8% accurate; ROC-AUC measures how well the model separates higher-risk from lower-risk cases.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limitations: This assignment uses March 2026 as a one-month feature window and April 2026 as the future outcome window, so it does not yet capture the longer 90-day history planned for the final project. Client histories are also unbalanced, meaning different clients may have different amounts of usable data. Search Console and GA4 availability is incomplete, so unavailable measurements must not be treated as real zeros. The 30% decline threshold is provisional and should be tested with alternative thresholds later. These results are predictive and directional, not causal.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.